# Computação Gráfica - Projeto 1
## Cena: Astronauta no Espaço

### Felipe da Costa Coqueiro, NUSP: 11781361
### Fernando Alee Suaiden, NUSP: 12680836

### Objetos:
1. **Astronauta** (3D) - Composto por esferas e cilindros
2. **Foguete** (3D) - Composto por cone e cilindros
3. **Sol** (2D) - Composto por triângulos
4. **Planeta** (3D) - Esfera com anéis (tipo Saturno)
5. **Satélite** (2D) - Composto por retângulos e painéis solares

## 1. Instalação das Dependências

In [37]:
# Instalar bibliotecas necessárias
!pip install PyOpenGL glfw numpy

## 2. Importações e Configurações Globais

In [38]:
import glfw
from OpenGL.GL import *
import numpy as np
import math
import random
import ctypes

# Dimensões da janela
WINDOW_WIDTH = 1200
WINDOW_HEIGHT = 800

# Estado do programa
wireframe_mode = False

# Transformações dos objetos
astronauta_pos = [0.0, 0.0, 0.0]      # Posição do astronauta (translação)
foguete_scale = 1.0                    # Escala do foguete
estrela_rotation = 0.0                 # Rotação da estrela (em radianos)
planeta_rotation = 0.0                 # Rotação automática do planeta (em radianos)

# Velocidades
TRANSLATION_SPEED = 0.05
SCALE_SPEED = 0.05
ROTATION_SPEED = 0.05  # em radianos

print("Configurações carregadas com sucesso!")

Configurações carregadas com sucesso!


## 3. Inicializando Janela

In [39]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE)
window = glfw.create_window(WINDOW_WIDTH, WINDOW_HEIGHT, "Computação Gráfica - Astronauta no Espaço", None, None)

if window is None:
    print("Falha ao criar janela GLFW")
    glfw.terminate()

glfw.make_context_current(window)

## 4. Shaders GLSL

Usamos `attribute` e `uniform` no padrão da aula:
- **Vertex Shader**: Recebe posição via `attribute vec3 position` e transforma via `uniform mat4 mat_transformation`
- **Fragment Shader**: Recebe cor via `uniform vec4 color`

Sem texturas, sem iluminação, sem câmera.

In [40]:
vertex_code = """
        attribute vec3 position;
        uniform mat4 mat_transformation;
        void main(){
            gl_Position = mat_transformation * vec4(position, 1.0);
        }
        """

fragment_code = """
        uniform vec4 color;
        void main(){
            gl_FragColor = color;
        }
        """

print("Shaders definidos!")

Shaders definidos!


## 5. Compilação e Bindagem dos Shaders

In [41]:
# Requisita slots na GPU
program  = glCreateProgram()
vertex   = glCreateShader(GL_VERTEX_SHADER)
fragment = glCreateShader(GL_FRAGMENT_SHADER)

# Associa código-fonte
glShaderSource(vertex, vertex_code)
glShaderSource(fragment, fragment_code)

# Compila Vertex Shader
glCompileShader(vertex)
if not glGetShaderiv(vertex, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(vertex).decode()
    print(error)
    raise RuntimeError("Erro de compilação do Vertex Shader")

# Compila Fragment Shader
glCompileShader(fragment)
if not glGetShaderiv(fragment, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(fragment).decode()
    print(error)
    raise RuntimeError("Erro de compilação do Fragment Shader")

# Bindagem
glAttachShader(program, vertex)
glAttachShader(program, fragment)

# Linkagem
glLinkProgram(program)
if not glGetProgramiv(program, GL_LINK_STATUS):
    print(glGetProgramInfoLog(program))
    raise RuntimeError('Erro de linkagem')

# Usa o programa
glUseProgram(program)

print("Shaders compilados e bindados com sucesso!")

Shaders compilados e bindados com sucesso!


## 6. Funções de Criação de Geometrias

Criamos primitivas geométricas 3D e 2D a partir do zero, retornando listas de triângulos expandidos (sem índices, compatível com `glDrawArrays`):
- **Esfera**: Para capacete, planeta, luvas
- **Cilindro**: Para corpo, braços, pernas
- **Cone**: Para ponta do foguete, aletas
- **Torus**: Para anéis do planeta
- **Estrela 2D**: Para o sol
- **Retângulo**: Para painéis solares do satélite

In [42]:
def create_sphere_triangles(radius, slices, stacks):
    """Cria esfera 3D. Retorna lista de vértices (x,y,z) em triângulos expandidos."""
    grid = []
    for i in range(stacks + 1):
        phi = math.pi * i / stacks
        for j in range(slices + 1):
            theta = 2.0 * math.pi * j / slices
            x = radius * math.sin(phi) * math.cos(theta)
            y = radius * math.cos(phi)
            z = radius * math.sin(phi) * math.sin(theta)
            grid.append((x, y, z))
    triangles = []
    for i in range(stacks):
        for j in range(slices):
            p0 = i * (slices + 1) + j
            p1 = p0 + slices + 1
            triangles.extend([grid[p0], grid[p1], grid[p0 + 1]])
            triangles.extend([grid[p1], grid[p1 + 1], grid[p0 + 1]])
    return triangles

def create_cylinder_triangles(radius, height, slices):
    """Cria cilindro 3D. Retorna lista de vértices (x,y,z) em triângulos expandidos."""
    bottom = []
    top = []
    for j in range(slices + 1):
        theta = 2.0 * math.pi * j / slices
        x = radius * math.cos(theta)
        z = radius * math.sin(theta)
        bottom.append((x, -height/2, z))
        top.append((x, height/2, z))
    triangles = []
    for j in range(slices):
        triangles.extend([bottom[j], top[j], bottom[j+1]])
        triangles.extend([top[j], top[j+1], bottom[j+1]])
    center_bottom = (0, -height/2, 0)
    center_top = (0, height/2, 0)
    for j in range(slices):
        triangles.extend([center_bottom, bottom[j+1], bottom[j]])
    for j in range(slices):
        triangles.extend([center_top, top[j], top[j+1]])
    return triangles

def create_cone_triangles(radius, height, slices):
    """Cria cone 3D. Retorna lista de vértices (x,y,z) em triângulos expandidos."""
    apex = (0, height/2, 0)
    base = []
    for j in range(slices + 1):
        theta = 2.0 * math.pi * j / slices
        x = radius * math.cos(theta)
        z = radius * math.sin(theta)
        base.append((x, -height/2, z))
    triangles = []
    for j in range(slices):
        triangles.extend([apex, base[j], base[j+1]])
    center = (0, -height/2, 0)
    for j in range(slices):
        triangles.extend([center, base[j+1], base[j]])
    return triangles

def create_torus_triangles(major_radius, minor_radius, major_segs, minor_segs):
    """Cria torus (anel) 3D. Retorna lista de vértices (x,y,z) em triângulos expandidos."""
    grid = []
    for i in range(major_segs + 1):
        theta = 2.0 * math.pi * i / major_segs
        ct, st = math.cos(theta), math.sin(theta)
        for j in range(minor_segs + 1):
            phi = 2.0 * math.pi * j / minor_segs
            cp, sp = math.cos(phi), math.sin(phi)
            x = (major_radius + minor_radius * cp) * ct
            y = minor_radius * sp
            z = (major_radius + minor_radius * cp) * st
            grid.append((x, y, z))
    triangles = []
    for i in range(major_segs):
        for j in range(minor_segs):
            p0 = i * (minor_segs + 1) + j
            p1 = p0 + minor_segs + 1
            triangles.extend([grid[p0], grid[p1], grid[p0 + 1]])
            triangles.extend([grid[p1], grid[p1 + 1], grid[p0 + 1]])
    return triangles

def create_star_2d_triangles(outer_radius, inner_radius, points):
    """Cria estrela 2D. Retorna lista de vértices (x,y,z) em triângulos expandidos."""
    center = (0, 0, 0)
    pts = []
    for i in range(points * 2):
        angle = math.pi / 2 + i * math.pi / points
        r = outer_radius if i % 2 == 0 else inner_radius
        pts.append((r * math.cos(angle), r * math.sin(angle), 0))
    triangles = []
    for i in range(points * 2):
        next_i = (i + 1) % (points * 2)
        triangles.extend([center, pts[i], pts[next_i]])
    return triangles

def create_rectangle_triangles(width, height):
    """Cria retângulo 2D. Retorna lista de vértices (x,y,z) em triângulos expandidos."""
    hw, hh = width / 2, height / 2
    return [
        (-hw, -hh, 0), (hw, -hh, 0), (hw, hh, 0),
        (-hw, -hh, 0), (hw, hh, 0), (-hw, hh, 0)
    ]

print("Funções de geometria definidas!")

Funções de geometria definidas!


## 7. Funções de Matrizes de Transformação

Implementamos manualmente as matrizes de transformação geométrica como arrays flat de 16 floats (padrão da aula):
- **Translação**: Move objetos no espaço
- **Escala**: Redimensiona objetos
- **Rotação**: Gira objetos em torno dos eixos X, Y e Z
- **multiplica_matriz**: Combina transformações

In [43]:
def multiplica_matriz(a, b):
    m_a = a.reshape(4,4)
    m_b = b.reshape(4,4)
    m_c = np.dot(m_a, m_b)
    c = m_c.reshape(1,16)
    return c

def mat_identity():
    return np.array([1.0, 0.0, 0.0, 0.0,
                     0.0, 1.0, 0.0, 0.0,
                     0.0, 0.0, 1.0, 0.0,
                     0.0, 0.0, 0.0, 1.0], np.float32)

def mat_translation(tx, ty, tz):
    return np.array([1.0, 0.0, 0.0, tx,
                     0.0, 1.0, 0.0, ty,
                     0.0, 0.0, 1.0, tz,
                     0.0, 0.0, 0.0, 1.0], np.float32)

def mat_rotation_x(angulo):
    c = math.cos(angulo)
    s = math.sin(angulo)
    return np.array([1.0, 0.0, 0.0, 0.0,
                     0.0,   c,  -s,  0.0,
                     0.0,   s,   c,  0.0,
                     0.0, 0.0, 0.0, 1.0], np.float32)

def mat_rotation_y(angulo):
    c = math.cos(angulo)
    s = math.sin(angulo)
    return np.array([  c,  0.0,   s, 0.0,
                     0.0,  1.0, 0.0, 0.0,
                      -s,  0.0,   c, 0.0,
                     0.0,  0.0, 0.0, 1.0], np.float32)

def mat_rotation_z(angulo):
    c = math.cos(angulo)
    s = math.sin(angulo)
    return np.array([  c,   -s, 0.0, 0.0,
                       s,    c, 0.0, 0.0,
                     0.0,  0.0, 1.0, 0.0,
                     0.0,  0.0, 0.0, 1.0], np.float32)

def mat_scale(sx, sy, sz):
    return np.array([ sx, 0.0, 0.0, 0.0,
                     0.0,  sy, 0.0, 0.0,
                     0.0, 0.0,  sz, 0.0,
                     0.0, 0.0, 0.0, 1.0], np.float32)

print("Funções de matrizes definidas!")

Funções de matrizes definidas!


## 8. Construção da Cena

Construímos todos os objetos da cena e armazenamos em um único buffer de vértices. Cada parte guarda: posição inicial no buffer, quantidade de vértices, cor e transformação local.

In [ ]:
# Listas globais para vértices e partes da cena
all_positions = []
parts = []

def add_part(triangles, color, local_transform, group):
    """Adiciona triângulos ao buffer global e registra a parte."""
    start = len(all_positions)
    count = len(triangles)
    all_positions.extend(triangles)
    parts.append({
        'start': start,
        'count': count,
        'color': color,
        'local_tf': local_transform,
        'group': group
    })

# ASTRONAUTA - Composto por esferas e cilindros

cor_capacete = (0.9, 0.9, 0.95)
cor_traje = (0.95, 0.95, 0.95)
cor_visor = (0.2, 0.3, 0.5)
cor_detalhes = (0.8, 0.4, 0.1)
cor_mochila = (0.6, 0.6, 0.65)

# Capacete (esfera)
add_part(create_sphere_triangles(0.12, 16, 16), cor_capacete,
         mat_translation(0, 0.28, 0), 'astronauta')
# Visor (esfera menor na frente)
add_part(create_sphere_triangles(0.08, 12, 12), cor_visor,
         mat_translation(0, 0.28, 0.06), 'astronauta')
# Corpo (cilindro)
add_part(create_cylinder_triangles(0.1, 0.25, 16), cor_traje,
         mat_translation(0, 0.08, 0), 'astronauta')
# Mochila de oxigênio (cilindro)
add_part(create_cylinder_triangles(0.07, 0.2, 8), cor_mochila,
         mat_translation(0, 0.1, -0.1), 'astronauta')
# Braço esquerdo (cilindro rotacionado)
braco_e_tf = multiplica_matriz(mat_translation(-0.15, 0.1, 0), mat_rotation_z(math.radians(150)))
add_part(create_cylinder_triangles(0.035, 0.18, 12), cor_traje,
         braco_e_tf, 'astronauta')
# Braço direito (cilindro rotacionado)
braco_d_tf = multiplica_matriz(mat_translation(0.15, 0.1, 0), mat_rotation_z(math.radians(-150)))
add_part(create_cylinder_triangles(0.035, 0.18, 12), cor_traje,
         braco_d_tf, 'astronauta')
# Luva esquerda (esfera)
add_part(create_sphere_triangles(0.04, 10, 10), cor_detalhes,
         multiplica_matriz(braco_e_tf, mat_translation(0, 0.11, 0)), 'astronauta')
# Luva direita (esfera)
add_part(create_sphere_triangles(0.04, 10, 10), cor_detalhes,
         multiplica_matriz(braco_d_tf, mat_translation(0, 0.11, 0)), 'astronauta')
# Perna esquerda (cilindro)
add_part(create_cylinder_triangles(0.04, 0.2, 12), cor_traje,
         mat_translation(-0.05, -0.15, 0), 'astronauta')
# Perna direita (cilindro)
add_part(create_cylinder_triangles(0.04, 0.2, 12), cor_traje,
         mat_translation(0.05, -0.15, 0), 'astronauta')
# Bota esquerda (esfera)
add_part(create_sphere_triangles(0.045, 10, 10), cor_detalhes,
         mat_translation(-0.05, -0.28, 0.02), 'astronauta')
# Bota direita (esfera)
add_part(create_sphere_triangles(0.045, 10, 10), cor_detalhes,
         mat_translation(0.05, -0.28, 0.02), 'astronauta')

# FOGUETE - Composto por cone e cilindros
cor_corpo_fog = (0.85, 0.85, 0.9)
cor_ponta = (0.9, 0.2, 0.1)
cor_janela = (0.3, 0.5, 0.8)
cor_propulsor = (0.3, 0.3, 0.35)
cor_chama = (1.0, 0.6, 0.1)

# Ponta (cone)
add_part(create_cone_triangles(0.08, 0.15, 20), cor_ponta,
         mat_translation(0, 0.35, 0), 'foguete')
# Corpo principal (cilindro)
add_part(create_cylinder_triangles(0.08, 0.4, 20), cor_corpo_fog,
         mat_translation(0, 0.08, 0), 'foguete')
# Janela 1 (esfera)
add_part(create_sphere_triangles(0.03, 12, 12), cor_janela,
         mat_translation(0, 0.15, 0.075), 'foguete')
# Janela 2 (esfera)
add_part(create_sphere_triangles(0.025, 10, 10), cor_janela,
         mat_translation(0, 0.05, 0.078), 'foguete')
# Propulsor (cilindro)
add_part(create_cylinder_triangles(0.06, 0.1, 16), cor_propulsor,
         mat_translation(0, -0.17, 0), 'foguete')
# Chama (cone invertido)
chama_tf = multiplica_matriz(mat_translation(0, -0.28, 0), mat_rotation_x(math.radians(180)))
add_part(create_cone_triangles(0.05, 0.12, 16), cor_chama,
         chama_tf, 'foguete')
# Aleta 1 (cone lateral esquerdo)
aleta1_tf = multiplica_matriz(mat_translation(-0.11, -0.16, 0), mat_rotation_z(math.radians(130)))
add_part(create_cone_triangles(0.03, 0.12, 12), cor_ponta,
         aleta1_tf, 'foguete')
# Aleta 2 (cone lateral direito)
aleta2_tf = multiplica_matriz(mat_translation(0.11, -0.16, 0), mat_rotation_z(math.radians(-130)))
add_part(create_cone_triangles(0.03, 0.12, 12), cor_ponta,
         aleta2_tf, 'foguete')
# Aleta 3 (cone traseiro)
aleta3_tf = multiplica_matriz(mat_translation(0, -0.16, -0.11), mat_rotation_x(math.radians(-130)))
add_part(create_cone_triangles(0.03, 0.12, 12), cor_ponta,
         aleta3_tf, 'foguete')

# SOL (2D) - Composta por triângulos

cor_centro_sol = (1.0, 0.9, 0.3)
cor_raios = (1.0, 0.7, 0.1)

# Centro (esfera)
add_part(create_sphere_triangles(0.08, 16, 16), cor_centro_sol,
         mat_identity(), 'estrela')
# Raios externos (estrela 2D)
add_part(create_star_2d_triangles(0.2, 0.1, 8), cor_raios,
         mat_identity(), 'estrela')
# Raios internos (estrela 2D rotacionada)
add_part(create_star_2d_triangles(0.15, 0.08, 8), cor_centro_sol,
         mat_rotation_z(math.radians(22.5)), 'estrela')

# PLANETA - Esfera com anéis (tipo Saturno)

cor_planeta = (0.6, 0.4, 0.3)
cor_faixa = (0.7, 0.5, 0.35)
cor_anel = (0.8, 0.75, 0.6)

# Corpo (esfera)
add_part(create_sphere_triangles(0.18, 24, 24), cor_planeta,
         mat_identity(), 'planeta')
# Faixa decorativa (torus fino)
add_part(create_torus_triangles(0.18, 0.01, 24, 8), cor_faixa,
         mat_identity(), 'planeta')
# Anel externo (torus)
add_part(create_torus_triangles(0.3, 0.02, 32, 8), cor_anel,
         mat_rotation_x(math.radians(75)), 'planeta')
# Anel interno (torus)
add_part(create_torus_triangles(0.25, 0.015, 32, 8), cor_anel,
         mat_rotation_x(math.radians(75)), 'planeta')

# SATÉLITE (2D + 3D) - Retângulos e painéis solares

cor_corpo_sat = (0.7, 0.7, 0.75)
cor_painel = (0.2, 0.3, 0.6)
cor_antena = (0.9, 0.9, 0.9)
cor_detalhe = (1.0, 0.8, 0.2)

# Corpo (cilindro)
add_part(create_cylinder_triangles(0.04, 0.1, 12), cor_corpo_sat,
         mat_identity(), 'satelite')
# Painel solar esquerdo (retângulo)
add_part(create_rectangle_triangles(0.15, 0.06), cor_painel,
         mat_translation(-0.11, 0, 0), 'satelite')
# Painel solar direito (retângulo)
add_part(create_rectangle_triangles(0.15, 0.06), cor_painel,
         mat_translation(0.11, 0, 0), 'satelite')
# Estrutura painel esquerdo
add_part(create_rectangle_triangles(0.16, 0.005), cor_detalhe,
         mat_translation(-0.11, 0.03, 0.001), 'satelite')
# Estrutura painel direito
add_part(create_rectangle_triangles(0.16, 0.005), cor_detalhe,
         mat_translation(0.11, 0.03, 0.001), 'satelite')
# Antena (cone)
add_part(create_cone_triangles(0.015, 0.06, 10), cor_antena,
         mat_translation(0, 0.08, 0), 'satelite')
# Prato da antena (cone invertido)
prato_tf = multiplica_matriz(mat_translation(0, 0.06, 0), mat_rotation_x(math.radians(180)))
add_part(create_cone_triangles(0.025, 0.02, 12), cor_detalhe,
         prato_tf, 'satelite')

# ESTRELAS DE FUNDO

random.seed(42)
for _ in range(50):
    x = random.uniform(-0.85, 0.85)
    y = random.uniform(-0.85, 0.85)
    z = 0.5
    tamanho = random.uniform(0.005, 0.015)
    brilho = random.uniform(0.7, 1.0)
    cor = (brilho, brilho, random.uniform(0.8, 1.0) * brilho)
    add_part(create_sphere_triangles(tamanho, 6, 6), cor,
             mat_translation(x, y, z), 'fundo')

print(f"Cena construída: {len(parts)} partes, {len(all_positions)} vértices no total.")

Cena construída: 85 partes, 28404 vértices no total.


## 9. Envio dos Dados à GPU

Enviamos todos os vértices para a GPU em um único buffer (VBO), seguindo o padrão da aula.

In [45]:
# Cria array estruturado de vértices 
num_vertices = len(all_positions)
vertices = np.zeros(num_vertices, [("position", np.float32, 3)])
vertices['position'] = all_positions

# Requisita buffer na GPU e envia dados
buffer_VBO = glGenBuffers(1)
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)
glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_STATIC_DRAW)
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)

# Configura atributo de posição
stride = vertices.strides[0]
offset = ctypes.c_void_p(0)

loc_position = glGetAttribLocation(program, "position")
glEnableVertexAttribArray(loc_position)
glVertexAttribPointer(loc_position, 3, GL_FLOAT, False, stride, offset)

# Pega localização dos uniforms
loc_color = glGetUniformLocation(program, "color")
loc_transformation = glGetUniformLocation(program, "mat_transformation")

print(f"Total de vértices enviados à GPU: {num_vertices}")

Total de vértices enviados à GPU: 28404


## 10. Eventos de Teclado

In [46]:
def key_event(window, key, scancode, action, mods):
    global wireframe_mode, astronauta_pos, foguete_scale, estrela_rotation

    # Translação do astronauta (WASD)
    if key == glfw.KEY_W: astronauta_pos[1] += TRANSLATION_SPEED
    if key == glfw.KEY_S: astronauta_pos[1] -= TRANSLATION_SPEED
    if key == glfw.KEY_A: astronauta_pos[0] -= TRANSLATION_SPEED
    if key == glfw.KEY_D: astronauta_pos[0] += TRANSLATION_SPEED

    # Escala do foguete (Z/X)
    if key == glfw.KEY_Z: foguete_scale = max(0.3, foguete_scale - SCALE_SPEED)
    if key == glfw.KEY_X: foguete_scale = min(2.0, foguete_scale + SCALE_SPEED)

    # Rotação da estrela (Q/E)
    if key == glfw.KEY_Q: estrela_rotation += ROTATION_SPEED
    if key == glfw.KEY_E: estrela_rotation -= ROTATION_SPEED

    # Toggle wireframe (P)
    if key == glfw.KEY_P and action == glfw.PRESS:
        wireframe_mode = not wireframe_mode

    # Sair (ESC)
    if key == glfw.KEY_ESCAPE: glfw.set_window_should_close(window, True)

glfw.set_key_callback(window, key_event)

## 11. Exibir Janela

In [47]:
glfw.show_window(window)

glEnable(GL_DEPTH_TEST) ### importante para 3D

## 12. Loop Principal

In [ ]:
while not glfw.window_should_close(window):

    glfw.poll_events()

    # Rotação automática do planeta
    planeta_rotation += 0.005

    glClearColor(0.02, 0.02, 0.08, 1.0)  # Fundo azul escuro (espaço)

    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)

    # Configura modo de renderização
    if wireframe_mode:
        glPolygonMode(GL_FRONT_AND_BACK, GL_LINE)
    else:
        glPolygonMode(GL_FRONT_AND_BACK, GL_FILL)

    # Inclinação da cena (reforça percepção 3D)
    mat_tilt = multiplica_matriz(
        mat_rotation_y(math.radians(-20)),
        mat_rotation_x(math.radians(12))
    )

    # ============================================================
    # Transformações por grupo de objetos
    # ============================================================
    world_transforms = {}

    # Estrelas de fundo (estáticas, apenas tilt)
    world_transforms['fundo'] = mat_tilt

    # Astronauta (translação com WASD)
    mat_astro = multiplica_matriz(
        mat_tilt,
        mat_translation(astronauta_pos[0] - 0.35, astronauta_pos[1], astronauta_pos[2])
    )
    world_transforms['astronauta'] = mat_astro

    # Foguete (escala com Z/X)
    mat_fog = multiplica_matriz(
        mat_translation(0.5, -0.2, 0.1),
        multiplica_matriz(mat_rotation_z(math.radians(-15)), mat_scale(foguete_scale, foguete_scale, foguete_scale))
    )
    mat_fog = multiplica_matriz(mat_tilt, mat_fog)
    world_transforms['foguete'] = mat_fog

    # Estrela/Sol (rotação com Q/E)
    mat_est = multiplica_matriz(
        mat_translation(-0.55, 0.5, 0.3),
        mat_rotation_z(estrela_rotation)
    )
    mat_est = multiplica_matriz(mat_tilt, mat_est)
    world_transforms['estrela'] = mat_est

    # Planeta (rotação automática)
    mat_pla = multiplica_matriz(
        mat_translation(0.6, 0.4, 0.2),
        multiplica_matriz(mat_rotation_y(planeta_rotation), mat_scale(0.6, 0.6, 0.6))
    )
    mat_pla = multiplica_matriz(mat_tilt, mat_pla)
    world_transforms['planeta'] = mat_pla

    # Satélite (estático)
    mat_sat = multiplica_matriz(
        mat_translation(-0.55, -0.45, 0.05),
        mat_rotation_z(math.radians(25))
    )
    mat_sat = multiplica_matriz(mat_tilt, mat_sat)
    world_transforms['satelite'] = mat_sat

    # Desenha todos os objetos

    for part in parts:
        world = world_transforms[part['group']]
        combined = multiplica_matriz(world, part['local_tf'])

        glUniformMatrix4fv(loc_transformation, 1, GL_TRUE, combined)
        glUniform4f(loc_color, part['color'][0], part['color'][1], part['color'][2], 1.0)
        glDrawArrays(GL_TRIANGLES, part['start'], part['count'])

    glfw.swap_buffers(window)

glfw.terminate()
print("Programa encerrado!")

Programa encerrado!


---

## Resumo dos Requisitos Atendidos

| Requisito | Implementação |
|-----------|---------------|
| 5+ objetos diferentes | Astronauta, Foguete, Estrela/Sol, Planeta, Satélite (+ estrelas de fundo) |
| 2+ objetos 3D | Astronauta (esferas+cilindros), Foguete (cone+cilindros), Planeta (esfera+torus) |
| Objetos compostos | Todos são composições de primitivas simples criadas manualmente |
| Sem projection/model/view | Única `mat_transformation` combinada (padrão aula) |
| Translação | Astronauta (teclas WASD) |
| Escala | Foguete (teclas Z/X) |
| Rotação | Estrela/Sol (teclas Q/E); Planeta (automática) |
| Visualizar malha (wireframe) | Tecla P |
| Padrão da aula | `attribute`/`uniform`, VBO, `glDrawArrays`, `multiplica_matriz`, arrays flat 16 floats |
| Sem funções deprecated | Nenhuma função obsoleta utilizada |